# MINT tutorial: RGB, depth, and instance overlay

This notebook uses only data embedded in one MINT file. It selects a random dynamic camera from `scene.cameras`, renders the first 30 RGB and depth frames, writes two MP4 files, and displays them inline. It then renders every instance label at frame 0 and draws the visible labels as a color overlay on RGB.

## Dependencies

The companion `requirements.txt` lists every direct dependency of this tutorial:

| Package | Purpose |
| --- | --- |
| `graciasdk` | Load and render the MINT on the GPU |
| `numpy` | Process RGB, depth, and coverage arrays |
| `av` | Write H.264 MP4 files |
| `matplotlib` | Draw and save the instance overlay |
| `jupyterlab`, `ipykernel`, `ipython` | Run the notebook and display videos inline |

From `/home/ubuntu/python-sdk`, start an isolated environment with one command:

```bash
uv run --isolated --no-project --python 3.12 --find-links wheels --with-requirements examples/notebooks/requirements.txt jupyter lab examples/notebooks/mint_rgb_depth_instances.ipynb
```

`uv` selects the local SDK wheel for the host platform and caches the resolved environment. `--isolated --no-project` prevents project discovery and changes to the repository environment. The notebook contains no installation cell and never mutates its active kernel.

In [ ]:
from importlib.metadata import version
from fractions import Fraction
from pathlib import Path
import random

import av
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, Video, display
from matplotlib.patches import Patch

from graciasdk import GraciaSDK

{
    package: version(package)
    for package in ("graciasdk", "numpy", "av", "matplotlib")
}

## Settings

Change `SEED` to select another embedded dynamic camera. The current MINT contains two dynamic cameras: `014` and `063`.

In [ ]:
MINT_PATH = Path("/data/ranges/exp/ranges_0_416_x86_64_cameras.mint")
OUTPUT_DIR = Path("/data/ranges/exp/tutorial_sdk_r1")

FRAME_COUNT = 30
FPS = 25.0
WIDTH = 800
HEIGHT = 670
SEED = 2026
MASK_THRESHOLD = 1.0 / 255.0

if not MINT_PATH.is_file():
    raise FileNotFoundError(MINT_PATH)
if WIDTH % 2 or HEIGHT % 2:
    raise ValueError("H.264 requires even WIDTH and HEIGHT")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Small rendering helpers

The depth buffer stores negative view-space Z. The notebook uses one depth range for all requested frames, then multiplies normalized depth by coverage to keep low-opacity edges faint. After each seek, the rendering helper waits until the requested scene time is buffered. It moves video time forward by one floating-point step to avoid an empty render on an exact decoder boundary. The camera stays at the requested time.

In [ ]:
def rgb_u8(color):
    rgb = np.clip(np.asarray(color)[..., :3].astype(np.float32), 0.0, 1.0)
    return np.rint(rgb * 255.0).astype(np.uint8)


def render_ready(scene, sdk, camera_track, timestamp, fps, timeout=30.0):
    scene_time = float(np.nextafter(timestamp, np.inf))
    scene.set_time(scene_time)
    if not scene.wait_buffered(timeout=timeout):
        raise TimeoutError(f"Frame at {timestamp:.3f}s stayed buffered")
    camera = camera_track.at(timestamp, fps)
    return sdk.render(scene, camera)


def make_depth_frames(depths, coverages):
    valid_masks = [
        (coverage > 0.0) & np.isfinite(depth) & (depth != 0.0)
        for depth, coverage in zip(depths, coverages)
    ]
    if not any(mask.any() for mask in valid_masks):
        raise ValueError("The rendered depth is empty")

    far = min(float(depth[mask].min()) for depth, mask in zip(depths, valid_masks) if mask.any())
    near = max(float(depth[mask].max()) for depth, mask in zip(depths, valid_masks) if mask.any())
    scale = max(near - far, np.finfo(np.float32).eps)
    frames = []

    for depth, coverage, valid in zip(depths, coverages, valid_masks):
        gray = np.zeros_like(depth, dtype=np.float32)
        gray[valid] = (depth[valid] - far) / scale
        gray *= coverage
        gray_u8 = np.rint(np.clip(gray, 0.0, 1.0) * 255.0).astype(np.uint8)
        frames.append(np.repeat(gray_u8[..., None], 3, axis=2))

    return frames, (far, near)


def write_mp4(path, frames, fps):
    if not frames:
        raise ValueError("No frames to write")

    height, width, channels = frames[0].shape
    if channels != 3:
        raise ValueError("Expected RGB frames")

    rate = Fraction(str(fps)).limit_denominator(100_000)
    time_base = Fraction(rate.denominator, rate.numerator)

    with av.open(str(path), "w", options={"movflags": "+faststart"}) as container:
        stream = container.add_stream("libx264", rate=rate)
        stream.width = width
        stream.height = height
        stream.pix_fmt = "yuv420p"
        stream.options = {"crf": "18", "preset": "medium"}

        for index, image in enumerate(frames):
            frame = av.VideoFrame.from_ndarray(np.ascontiguousarray(image), format="rgb24")
            frame.pts = index
            frame.time_base = time_base
            for packet in stream.encode(frame):
                container.mux(packet)

        for packet in stream.encode():
            container.mux(packet)

## Load the MINT and select a dynamic camera

`scene.cameras` comes from the embedded `CAMERAS` chunk. This notebook does not read an external camera file.

In [ ]:
sdk = GraciaSDK(WIDTH, HEIGHT)
scene = sdk.load(str(MINT_PATH))
scene.wait_ready(timeout=120.0)

dynamic_cameras = [
    camera
    for camera in scene.cameras
    if camera.is_dynamic and camera.poses_count >= FRAME_COUNT
]
if not dynamic_cameras:
    raise ValueError("The MINT has no dynamic camera with enough poses")

camera_track = random.Random(SEED).choice(dynamic_cameras)
print(f"Embedded cameras: {len(scene.cameras)}")
print(f"Dynamic cameras: {[camera.name for camera in dynamic_cameras]}")
print(f"Selected camera: {camera_track.name} ({camera_track.poses_count} poses)")

## Render the first 30 frames

In [ ]:
scene.set_uint("instance_filter", 0)
rgb_frames = []
depth_values = []
coverage_values = []

for frame_index in range(FRAME_COUNT):
    timestamp = frame_index / FPS
    result = render_ready(scene, sdk, camera_track, timestamp, FPS)
    rgb_frames.append(rgb_u8(result.color))
    depth_values.append(np.asarray(result.depth, dtype=np.float32).copy())
    coverage_values.append(np.asarray(result.coverage, dtype=np.float32).copy())
    if frame_index == 0 or (frame_index + 1) % 5 == 0:
        print(f"Rendered {frame_index + 1}/{FRAME_COUNT}")

## Write and view RGB and depth video

The cell embeds both short videos in its output, so they play directly in Jupyter.

In [ ]:
depth_frames, (depth_far, depth_near) = make_depth_frames(depth_values, coverage_values)
rgb_video = OUTPUT_DIR / f"camera_{camera_track.name}_first_{FRAME_COUNT}_rgb.mp4"
depth_video = OUTPUT_DIR / f"camera_{camera_track.name}_first_{FRAME_COUNT}_depth.mp4"

write_mp4(rgb_video, rgb_frames, FPS)
write_mp4(depth_video, depth_frames, FPS)

print(f"RGB: {rgb_video}")
print(f"Depth: {depth_video}")
print(f"Shared depth range: far={depth_far:.4f}, near={depth_near:.4f}")

display(Markdown("### RGB"))
display(Video(filename=str(rgb_video), embed=True, width=WIDTH, html_attributes="controls loop muted playsinline"))
display(Markdown("### Depth"))
display(Video(filename=str(depth_video), embed=True, width=WIDTH, html_attributes="controls loop muted playsinline"))

## Render every instance label at frame 0

The SDK renders the selected instance in white and every other splat in black. All splats still take part in visibility, so foreground objects occlude the selected instance. The mean RGB highlight is the soft instance mask. `result.coverage` describes the full scene and is not an instance mask. The code keeps the strongest highlight at each pixel and uses its strength for smooth overlay edges.

In [ ]:
scene.set_uint("instance_filter", 0)
first_result = render_ready(scene, sdk, camera_track, 0.0, FPS)
first_rgb = rgb_u8(first_result.color)
first_camera = camera_track.at(0.0, FPS)
instance_names = {int(key): value for key, value in scene.instance_names().items()}
instance_ids = sorted(instance_names)

label_map = np.zeros((HEIGHT, WIDTH), dtype=np.int32)
best_highlight = np.zeros((HEIGHT, WIDTH), dtype=np.float32)

for index, instance_id in enumerate(instance_ids, start=1):
    scene.set_uint("instance_filter", instance_id)
    result = sdk.render(scene, first_camera)
    highlight_rgb = np.asarray(result.color)[..., :3].astype(np.float32)
    highlight = highlight_rgb.mean(axis=-1)
    take = (highlight >= MASK_THRESHOLD) & (highlight > best_highlight)
    label_map[take] = instance_id
    best_highlight[take] = highlight[take]
    if index % 10 == 0 or index == len(instance_ids):
        print(f"Rendered labels {index}/{len(instance_ids)}")

scene.set_uint("instance_filter", 0)
visible_ids = [instance_id for instance_id in instance_ids if np.any(label_map == instance_id)]
positions = np.linspace(0.05, 0.95, len(instance_ids))
colors = {
    instance_id: np.asarray(plt.colormaps["turbo"](position)[:3])
    for instance_id, position in zip(instance_ids, positions)
}

overlay = first_rgb.astype(np.float32) / 255.0
for instance_id in visible_ids:
    mask = label_map == instance_id
    alpha = (0.55 * best_highlight[mask])[:, None]
    overlay[mask] = (1.0 - alpha) * overlay[mask] + alpha * colors[instance_id]

fig, (rgb_axis, overlay_axis) = plt.subplots(1, 2, figsize=(18, 8))
rgb_axis.imshow(first_rgb)
rgb_axis.set_title(f"RGB · camera {camera_track.name} · frame 0")
rgb_axis.axis("off")

overlay_axis.imshow(np.clip(overlay, 0.0, 1.0))
overlay_axis.set_title(f"Instance overlay · {len(visible_ids)}/{len(instance_ids)} visible")
overlay_axis.axis("off")

handles = [
    Patch(color=colors[instance_id], label=f"{instance_id}: {instance_names[instance_id]}")
    for instance_id in visible_ids
]
if handles:
    overlay_axis.legend(handles=handles, loc="upper left", bbox_to_anchor=(1.01, 1.0), fontsize=8)

overlay_image = OUTPUT_DIR / f"camera_{camera_track.name}_frame_000_instances.png"
fig.tight_layout()
fig.savefig(overlay_image, dpi=150, bbox_inches="tight")
plt.show()

print(f"Rendered all {len(instance_ids)} labels")
print("Visible labels:", [f"{instance_id}:{instance_names[instance_id]}" for instance_id in visible_ids])
print(f"Overlay: {overlay_image}")